# Extracción de datos desde IGDB

En este notebook se realiza la extracción de datos crudos desde la API de IGDB
para el proyecto de análisis de videojuegos (indie vs AAA, géneros, plataformas, etc.).

Los resultados se guardan en archivos `.parquet` dentro de la carpeta `data/`.


In [ ]:
import time
from pathlib import Path
import requests
import pandas as pd

In [3]:
client_id = "lbosp4quzjq9oe0nhgd7k291j7lf7y"
client_secret = "ztxmpntq9g85h6mc6uju1v0mijq4ap"  

url = "https://id.twitch.tv/oauth2/token"
params = {
    "client_id": client_id,
    "client_secret": client_secret,
    "grant_type": "client_credentials"
}

resp = requests.post(url, params=params)
print(resp.json())


{'access_token': '4miv6lwnd2bdwz3mu87cs59pxrmmdf', 'expires_in': 5468327, 'token_type': 'bearer'}


In [ ]:
ruta_data = Path("data")
ruta_data.mkdir(exist_ok=True)

# Credenciales de IGDB/Twitch
# Idealmente cargarlas desde variables de entorno o un archivo separado
CLIENT_ID = "lbosp4quzjq9oe0nhgd7k291j7lf7y"
ACCESS_TOKEN = "4miv6lwnd2bdwz3mu87cs59pxrmmdf"


# URL base de la API
URL_BASE_IGDB = "https://api.igdb.com/v4"

In [5]:
# =========================== #
#  Funciones auxiliares IGDB  #
# =========================== #

def construir_encabezados():
    """Devuelve los encabezados necesarios para llamar a la API de IGDB."""
    return {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {ACCESS_TOKEN}",
    }


def hacer_consulta_igdb(endpoint, consulta):
    """
    Realiza una consulta POST al endpoint dado de IGDB
    usando la sintaxis de APICalypse en el cuerpo.
    
    endpoint: str, por ejemplo "games", "genres"
    consulta: str, por ejemplo "fields id,name; limit 10;"
    """
    url = f"{URL_BASE_IGDB}/{endpoint}"
    respuesta = requests.post(url, headers=construir_encabezados(), data=consulta)
    
    if not respuesta.ok:
        print(f"Error {respuesta.status_code} al consultar {endpoint}: {respuesta.text[:200]}")
        respuesta.raise_for_status()
    
    return respuesta.json()


def descargar_paginado(endpoint, consulta_base, limite=500, max_paginas=None, pausa_segundos=0.4):
    """
    Descarga resultados en páginas usando 'offset' y 'limit'.
    
    - endpoint: str, por ejemplo "games"
    - consulta_base: str, sin 'limit' ni 'offset' al final
    - limite: int, cantidad de filas por página (máx. recomendado 500)
    - max_paginas: int o None, para cortar la descarga en pruebas
    - pausa_segundos: float, para no pegarle tan seguido a la API
    """
    todos_los_resultados = []
    offset = 0
    numero_pagina = 0
    
    while True:
        numero_pagina += 1
        if max_paginas is not None and numero_pagina > max_paginas:
            break
        
        consulta = (
            consulta_base
            + f"\nlimit {limite};"
            + f"\noffset {offset};"
        )
        
        datos = hacer_consulta_igdb(endpoint, consulta)
        
        if not datos:
            # No llegaron más resultados, cortamos el loop
            break
        
        todos_los_resultados.extend(datos)
        
        offset += limite
        time.sleep(pausa_segundos)
    
    return todos_los_resultados


In [11]:
consulta_juegos_base = """
fields
    id,
    name,
    slug,
    first_release_date,
    game_type,
    status,
    total_rating,
    total_rating_count,
    rating,
    rating_count,
    aggregated_rating,
    aggregated_rating_count,
    hypes,
    follows,
    genres,
    themes,
    keywords,
    game_modes,
    player_perspectives,
    platforms,
    involved_companies;
sort first_release_date desc;
"""


juegos_raw = descargar_paginado(
    endpoint="games",
    consulta_base=consulta_juegos_base,
    limite=500,
    max_paginas=1000,
    pausa_segundos=0.4,
)

len(juegos_raw)



343323

In [ ]:
df_juegos_raw = pd.json_normalize(juegos_raw)
descarga = df_juegos_raw.to_csv(ruta_data / "juegos_raw.csv", index=False)


In [14]:
df_juegos_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343323 entries, 0 to 343322
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       343323 non-null  int64  
 1   first_release_date       251244 non-null  float64
 2   game_modes               218031 non-null  object 
 3   genres                   284979 non-null  object 
 4   hypes                    17946 non-null   float64
 5   involved_companies       164124 non-null  object 
 6   name                     343323 non-null  object 
 7   platforms                275067 non-null  object 
 8   slug                     343323 non-null  object 
 9   themes                   193964 non-null  object 
 10  game_type                343323 non-null  int64  
 11  keywords                 128585 non-null  object 
 12  player_perspectives      134703 non-null  object 
 13  status                   25427 non-null   float64
 14  rati

In [16]:
consulta_generos = """
fields id,name,slug;
limit 500;
"""

generos_raw = hacer_consulta_igdb("genres", consulta_generos)

In [ ]:
df_generos = pd.json_normalize(generos_raw)
df_generos.to_csv(ruta_data / "igdb_generos.csv", index=False)
df_generos.head()

,id,name,slug
0,2,Point-and-click,point-and-click
1,4,Fighting,fighting
2,5,Shooter,shooter
3,7,Music,music
4,8,Platform,platform


In [18]:
df_generos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      23 non-null     int64 
 1   name    23 non-null     object
 2   slug    23 non-null     object
dtypes: int64(1), object(2)
memory usage: 684.0+ bytes


In [19]:
# ========================== #
#  Descarga de /platforms    #
# ========================== #

consulta_plataformas = """
fields
    id,
    name,
    slug,
    platform_family;
limit 500;
"""


In [20]:
plataformas_raw = hacer_consulta_igdb("platforms", consulta_plataformas)
df_plataformas = pd.json_normalize(plataformas_raw)
df_plataformas.to_csv(ruta_data / "igdb_plataformas.csv", index=False)
df_plataformas.head()

,id,name,slug,platform_family
0,376,Epoch Super Cassette Vision,epoch-super-cassette-vision,NaN
1,510,e-Reader / Card-e Reader,e-reader-slash-card-e-reader,5.0
2,12,Xbox 360,xbox360,2.0
3,32,Sega Saturn,saturn,3.0
4,62,Atari Jaguar,jaguar,NaN


In [21]:
df_plataformas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               220 non-null    int64  
 1   name             220 non-null    object 
 2   slug             220 non-null    object 
 3   platform_family  55 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 7.0+ KB


In [22]:
# ===================================== #
#  Descarga de /involved_companies      #
# ===================================== #

# Si ya tienes df_juegos_raw en memoria, usamos eso.
# Si no, lo puedes leer desde el CSV:
# df_juegos_raw = pd.read_csv(ruta_data / "igdb_juegos_raw.csv")

ids_juegos = df_juegos_raw["id"].dropna().unique().tolist()

tamaño_batch = 500
registros_involved = []

for i in range(0, len(ids_juegos), tamaño_batch):
    batch_ids = ids_juegos[i : i + tamaño_batch]
    lista_ids_str = ",".join(str(x) for x in batch_ids)
    
    consulta_involved = f"""
    fields
        id,
        game,
        company,
        developer,
        publisher,
        porting,
        supporting;
    where game = ({lista_ids_str});
    """
    
    datos_batch = descargar_paginado(
        endpoint="involved_companies",
        consulta_base=consulta_involved,
        limite=500,
        max_paginas=None,   # dejamos que se pagine solo si hace falta
        pausa_segundos=0.4,
    )
    
    registros_involved.extend(datos_batch)


In [23]:
df_involved = pd.json_normalize(registros_involved)
df_involved.to_csv(ruta_data / "igdb_involved_companies.csv", index=False)
df_involved.head()

,id,company,developer,game,porting,publisher,supporting
0,331112,65568,True,355842,False,True,False
1,332848,65948,True,356349,False,True,False
2,332896,65957,True,356337,False,True,False
3,328047,64418,True,347911,False,True,False
4,329079,64808,False,348180,False,True,False


In [24]:
df_involved.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250575 entries, 0 to 250574
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   id          250575 non-null  int64
 1   company     250575 non-null  int64
 2   developer   250575 non-null  bool 
 3   game        250575 non-null  int64
 4   porting     250575 non-null  bool 
 5   publisher   250575 non-null  bool 
 6   supporting  250575 non-null  bool 
dtypes: bool(4), int64(3)
memory usage: 6.7 MB


In [25]:
# ========================= #
#  Descarga de /companies   #
# ========================= #

# Ids únicos de empresas que participan en nuestros juegos
ids_empresas = (
    df_involved["company"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

tamaño_batch_empresas = 500
registros_empresas = []

for i in range(0, len(ids_empresas), tamaño_batch_empresas):
    batch_ids_emp = ids_empresas[i : i + tamaño_batch_empresas]
    lista_ids_emp_str = ",".join(str(x) for x in batch_ids_emp)
    
    consulta_empresas = f"""
    fields
        id,
        name,
        slug,
        country,
        start_date,
        parent;
    where id = ({lista_ids_emp_str});
    """
    
    datos_empresas_batch = hacer_consulta_igdb("companies", consulta_empresas)
    registros_empresas.extend(datos_empresas_batch)



In [26]:
df_empresas = pd.json_normalize(registros_empresas)
df_empresas.to_csv(ruta_data / "igdb_empresas.csv", index=False)
df_empresas.head()

,id,country,name,slug,start_date,parent
0,26,392.0,Square Enix,square-enix,1.049155e+09,NaN
1,29,840.0,Rockstar Games,rockstar-games,9.124704e+08,139.0
2,37,392.0,Capcom,capcom,2.968704e+08,NaN
3,38,124.0,Ubisoft Montreal,ubisoft-montreal,8.598528e+08,104.0
4,50,840.0,WB Games,wb-games,7.572960e+08,1633.0


In [ ]:
df_empresas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1130 entries, 0 to 1129
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          1130 non-null   int64  
 1   country     741 non-null    float64
 2   name        1130 non-null   object 
 3   slug        1130 non-null   object 
 4   start_date  623 non-null    float64
 5   parent      276 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 53.1+ KB


: 